# Copilot Audit Log - Processor 
**Purpose.** Move the heavy Power Query transformations off the Power BI refresh path.
This notebook does, once in Spark, everything the `Chat + Agent Interactions (Audit Logs)`
Power Query used to do row-by-row in the single-threaded mashup engine:

* parse `AccessedResources` / `AISystemPlugin` JSON and flatten them,
* explode the accessed-resources list (1 interaction -> N resource rows),
* derive `InteractionDate` / `WeekStart` / `MonthStart`,
* normalise the UPN and left-join the licence flag,
* resolve `Agent_LinkID` via the 3-way agent map (Entra id -> Title id -> name).

It writes a single, flat, V-Ordered Delta table **`copilot_interactions_curated`** whose
column set is identical to the old Power Query output, so every calculated column,
measure and relationship in the model keeps working unchanged.

Power BI then reads this table with **no transformation** (thin, foldable passthrough),
which is what makes both Direct Lake and a fast Incremental Refresh possible.

> Run this AFTER the audit-log ingester has produced `copilot_interactions_parsed`,
> and after the licensed-users and agents-365 producers have landed their tables.
> Schedule it in the same pipeline, immediately before the semantic-model refresh.

In [ ]:
# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
SRC_INTERACTIONS = "copilot_interactions_parsed"   # raw fact from the audit-log ingester
SRC_LICENSED     = "copilot_licensed_users"        # licensed-users dim (optional)
SRC_AGENTS       = "agents_365"                    # agents 365 dim (optional)

OUT_TABLE        = "copilot_interactions_curated"  # <-- Power BI reads this

# Full rebuild vs incremental append. For the initial backfill use "overwrite".
# For daily runs use "merge" (idempotent upsert on the natural key below).
WRITE_MODE       = "overwrite"                       # "overwrite" | "merge"
MERGE_KEYS       = ["Id"]                            # unique interaction id column(s) if present

RUN_OPTIMIZE     = True                              # OPTIMIZE + VORDER after write

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import DataFrame

# V-Order every Parquet/Delta file this session writes (required for Direct Lake perf).
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
# Let Spark evolve the Delta schema when tenant exports add/drop optional columns.
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")


def has_col(df: DataFrame, name: str) -> bool:
    return name in df.columns


def ensure_col(df: DataFrame, name: str, default=F.lit(None), dtype="string") -> DataFrame:
    # Idempotent 'add column if missing' -- mirrors the M Table.HasColumns guards.
    if has_col(df, name):
        return df
    return df.withColumn(name, default.cast(dtype))


def first_existing(df: DataFrame, candidates, fallback=None):
    for c in candidates:
        if c in df.columns:
            return c
    return fallback

In [ ]:
# ----------------------------------------------------------------------------
# 1. READ RAW FACT
# ----------------------------------------------------------------------------
fact = spark.table(SRC_INTERACTIONS)
print(f"{SRC_INTERACTIONS}: {fact.count():,} rows, {len(fact.columns)} cols")

In [ ]:
# ----------------------------------------------------------------------------
# 2. ENSURE OPTIONAL COLUMNS EXIST (tenant exports vary)
#    Mirrors the 'Add ...' idempotent steps in the Power Query.
# ----------------------------------------------------------------------------
fact = ensure_col(fact, "Message_isPrompt", F.lit("TRUE"))
fact = ensure_col(fact, "ModelTransparencyDetails_ModelProviderName")
fact = ensure_col(fact, "ModelTransparencyDetails_ModelName")
fact = ensure_col(fact, "ApplicationName")
fact = ensure_col(fact, "Audit_UserKey")
fact = ensure_col(fact, "SensitivityLabelId")
fact = ensure_col(fact, "AccessedResource_SensitivityLabelId")

# AppIdentity -> AppIdentity_AppId / AppIdentity_DisplayName, then drop the raw column.
fact = ensure_col(fact, "AppIdentity_AppId")
if has_col(fact, "AppIdentity"):
    if not has_col(fact, "AppIdentity_DisplayName"):
        fact = fact.withColumn("AppIdentity_DisplayName", F.col("AppIdentity").cast("string"))
    fact = fact.drop("AppIdentity")
else:
    fact = ensure_col(fact, "AppIdentity_DisplayName")

In [ ]:
# ----------------------------------------------------------------------------
# 3. ACCESSED RESOURCES  --  parse JSON + explode (this was the big fold-breaker)
#    Old M: Json.Document -> Table.ExpandListColumn -> Table.ExpandRecordColumn.
#    Here: from_json to an array<struct>, explode_outer, then project.
# ----------------------------------------------------------------------------
ar_schema = T.ArrayType(T.StructType([
    T.StructField("Type",    T.StringType()),
    T.StructField("Action",  T.StringType()),
    T.StructField("SiteUrl", T.StringType()),
]))

if has_col(fact, "AccessedResources"):
    parsed = fact.withColumn(
        "_resources",
        F.when(
            (F.col("AccessedResources").isNull()) | (F.trim(F.col("AccessedResources")) == ""),
            F.array().cast(ar_schema),
        ).otherwise(F.coalesce(F.from_json(F.col("AccessedResources"), ar_schema),
                               F.array().cast(ar_schema))),
    ).drop("AccessedResources")

    # explode_outer keeps interactions that have zero accessed resources (LEFT semantics).
    parsed = parsed.withColumn("_res", F.explode_outer("_resources")).drop("_resources")
    fact = (parsed
            .withColumn("AccessedResource_Type",    F.col("_res.Type"))
            .withColumn("AccessedResource_Action",  F.col("_res.Action"))
            .withColumn("AccessedResource_SiteUrl", F.col("_res.SiteUrl"))
            .drop("_res"))
else:
    fact = ensure_col(fact, "AccessedResource_Type")
    fact = ensure_col(fact, "AccessedResource_Action")
    fact = ensure_col(fact, "AccessedResource_SiteUrl")

In [ ]:
# ----------------------------------------------------------------------------
# 4. AI SYSTEM PLUGIN  --  parse JSON, take first element if it is a list
# ----------------------------------------------------------------------------
plugin_obj = T.StructType([
    T.StructField("Id",   T.StringType()),
    T.StructField("Name", T.StringType()),
])
plugin_arr = T.ArrayType(plugin_obj)

if has_col(fact, "AISystemPlugin"):
    raw = F.col("AISystemPlugin")
    as_arr = F.from_json(raw, plugin_arr)
    as_obj = F.from_json(raw, plugin_obj)
    single = F.when(as_arr.isNotNull() & (F.size(as_arr) > 0), as_arr.getItem(0)).otherwise(as_obj)
    fact = (fact
            .withColumn("_plugin", F.when((raw.isNull()) | (F.trim(raw) == ""), F.lit(None).cast(plugin_obj)).otherwise(single))
            .withColumn("AISystemPlugin_Id",   F.col("_plugin.Id"))
            .withColumn("AISystemPlugin_Name", F.col("_plugin.Name"))
            .drop("_plugin").drop("AISystemPlugin"))
else:
    fact = ensure_col(fact, "AISystemPlugin_Id")
    fact = ensure_col(fact, "AISystemPlugin_Name")

In [ ]:
# ----------------------------------------------------------------------------
# 5. DATES + RESOURCE COUNT
# ----------------------------------------------------------------------------
fact = fact.withColumn("CreationDate", F.to_timestamp("CreationDate"))

for c in ["InteractionDate", "WeekStart", "MonthStart"]:
    if has_col(fact, c):
        fact = fact.drop(c)

fact = (fact
        .withColumn("InteractionDate", F.to_date("CreationDate"))
        # Power Query used Day.Monday as the week start.
        .withColumn("WeekStart", F.date_sub(F.next_day(F.col("InteractionDate"), "Mon"), 7))
        .withColumn("MonthStart", F.trunc(F.col("InteractionDate"), "month")))

if has_col(fact, "Resource_Count"):
    fact = fact.withColumn("Resource_Count", F.col("Resource_Count").cast("long"))
else:
    fact = fact.withColumn("Resource_Count", F.lit(1).cast("long"))

In [ ]:
# ----------------------------------------------------------------------------
# 6. NORMALISED UPN  (join key to the licensed-users dim)
# ----------------------------------------------------------------------------
if has_col(fact, "Audit_UserId_Normalized"):
    norm = F.coalesce(F.col("Audit_UserId_Normalized"),
                      F.lower(F.trim(F.col("Audit_UserId").cast("string"))))
elif has_col(fact, "Audit_UserId"):
    norm = F.lower(F.trim(F.col("Audit_UserId").cast("string")))
else:
    norm = F.lit(None).cast("string")
fact = fact.withColumn("_NormUPN", norm)

In [ ]:
# ----------------------------------------------------------------------------
# 7. LEFT-JOIN LICENCE FLAG  (dedup dim first -> no row fan-out)
# ----------------------------------------------------------------------------
def load_licensed():
    try:
        lic = spark.table(SRC_LICENSED)
    except Exception as e:
        print(f"[warn] {SRC_LICENSED} not found ({e}); 'Has license' will be null.")
        return None
    upn_col = first_existing(lic, ["User Principal Name", "userPrincipalName",
                                   "UserPrincipalName", "User principal name",
                                   "User_Principal_Name"])
    has_lic = first_existing(lic, ["Has license", "Has License", "HasLicense", "HasCopilot",
                                   "Has Copilot", "Has Copilot License", "HasCopilotLicense",
                                   "isUser", "Has_license"])
    if has_col(lic, "UPN_Normalized"):
        key = F.col("UPN_Normalized")
    elif upn_col:
        key = F.lower(F.trim(F.col(upn_col).cast("string")))
    else:
        return None
    hl = F.col(has_lic).cast("string") if has_lic else F.lit("Unknown")
    lic = (lic.withColumn("UPN_Normalized", key)
              .withColumn("Has license", hl)
              .where(F.col("UPN_Normalized").isNotNull() & (F.trim("UPN_Normalized") != ""))
              .select("UPN_Normalized", "Has license")
              .dropDuplicates(["UPN_Normalized"]))
    return lic

lic = load_licensed()
if lic is not None:
    fact = fact.join(F.broadcast(lic), fact["_NormUPN"] == lic["UPN_Normalized"], "left") \
               .drop(lic["UPN_Normalized"])
else:
    fact = ensure_col(fact, "Has license")

In [ ]:
# ----------------------------------------------------------------------------
# 8. AGENT_TITLEID  (keep existing; else derive text before first '.' of AgentId)
# ----------------------------------------------------------------------------
aid = F.trim(F.col("AgentId").cast("string")) if has_col(fact, "AgentId") else F.lit(None).cast("string")
derived = F.when(aid.isNull() | (aid == ""), F.lit(None)) \
           .otherwise(F.when(F.instr(aid, ".") > 0, F.substring_index(aid, ".", 1)).otherwise(aid))
if has_col(fact, "Agent_TitleID"):
    existing = F.trim(F.col("Agent_TitleID").cast("string"))
    fact = fact.withColumn("Agent_TitleID",
                           F.when(existing.isNotNull() & (existing != ""), existing).otherwise(derived))
else:
    fact = fact.withColumn("Agent_TitleID", derived)

fact = ensure_col(fact, "Agent_EntraId")
fact = ensure_col(fact, "AgentName")

In [ ]:
# ----------------------------------------------------------------------------
# 9. THREE DEDUPED AGENT MAPS  (Entra id / Title id / normalised name -> Title ID)
#    then resolve Agent_LinkID = first non-null of Entra, Direct, Name.
# ----------------------------------------------------------------------------
def load_agents():
    try:
        return spark.table(SRC_AGENTS)
    except Exception as e:
        print(f"[warn] {SRC_AGENTS} not found ({e}); Agent_LinkID falls back to null.")
        return None

ag = load_agents()
fact = fact.withColumn("__nkey_fact", F.lower(F.trim(F.col("AgentName").cast("string"))))

if ag is not None and "Title ID" in ag.columns:
    entra_col = first_existing(ag, ["Entra Agent ID", "EntraAgentId", "Entra Agent Id",
                                    "EntraAgentID", "Agent ID", "AgentId", "Agent Id",
                                    "Bot Id", "BotId"])
    name_col  = first_existing(ag, ["Agent name", "Name"])

    title = F.trim(F.col("Title ID").cast("string"))
    ag2 = ag.withColumn("_title", title)

    entra_map = (ag2.withColumn("_entra", F.trim(F.col(entra_col).cast("string")) if entra_col else F.lit(None).cast("string"))
                    .where(F.col("_entra").isNotNull() & (F.col("_entra") != ""))
                    .select("_entra", "_title").dropDuplicates(["_entra"])) if entra_col else None
    title_map = (ag2.where(F.col("_title").isNotNull() & (F.col("_title") != ""))
                    .select(F.col("_title").alias("_tkey"), F.col("_title").alias("_title2"))
                    .dropDuplicates(["_tkey"]))
    name_map  = (ag2.withColumn("_nkey", F.lower(F.trim(F.col(name_col).cast("string"))) if name_col else F.lit(None).cast("string"))
                    .where(F.col("_nkey").isNotNull() & (F.col("_nkey") != ""))
                    .select("_nkey", "_title").dropDuplicates(["_nkey"])) if name_col else None

    if entra_map is not None:
        fact = fact.join(F.broadcast(entra_map), fact["Agent_EntraId"] == entra_map["_entra"], "left") \
                   .withColumnRenamed("_title", "__EntraTitle").drop("_entra")
    else:
        fact = fact.withColumn("__EntraTitle", F.lit(None).cast("string"))

    fact = fact.join(F.broadcast(title_map), fact["Agent_TitleID"] == title_map["_tkey"], "left") \
               .withColumnRenamed("_title2", "__DirectTitle").drop("_tkey")

    if name_map is not None:
        fact = fact.join(F.broadcast(name_map), fact["__nkey_fact"] == name_map["_nkey"], "left") \
                   .withColumnRenamed("_title", "__NameTitle").drop("_nkey")
    else:
        fact = fact.withColumn("__NameTitle", F.lit(None).cast("string"))
else:
    fact = fact.withColumn("__EntraTitle", F.lit(None).cast("string")) \
               .withColumn("__DirectTitle", F.lit(None).cast("string")) \
               .withColumn("__NameTitle", F.lit(None).cast("string"))

def _clean(c):
    t = F.trim(F.col(c).cast("string"))
    return F.when(t.isNotNull() & (t != ""), t)

fact = fact.withColumn("Agent_LinkID",
                       F.coalesce(_clean("__EntraTitle"), _clean("__DirectTitle"), _clean("__NameTitle")))
fact = fact.drop("__EntraTitle", "__DirectTitle", "__NameTitle", "__nkey_fact",
                 "_NormUPN", "Audit_UserId_Normalized")

In [ ]:
# ----------------------------------------------------------------------------
# 10. GUARANTEE THE MODEL CONTRACT
#     The semantic model binds to these 33 sourced columns. A tenant export can
#     omit optional ones, so add any missing as typed nulls -- the import then
#     never errors 'column not found', exactly like the old Power Query guards.
# ----------------------------------------------------------------------------
REQUIRED_TEXT_COLS = [
    "AISystemPlugin_Id", "AISystemPlugin_Name",
    "AccessedResource_Action", "AccessedResource_SensitivityLabelId",
    "AccessedResource_SiteUrl", "AccessedResource_Type",
    "AgentId", "AgentName", "Agent_EntraId", "Agent_LinkID", "Agent_TitleID",
    "AppHost", "AppIdentity_AppId", "AppIdentity_DisplayName", "AppIdentity_PublisherId",
    "ApplicationName", "Audit_UserId", "Audit_UserKey", "ClientRegion", "Context_Type",
    "Has license", "Message_Id", "Message_isPrompt",
    "ModelTransparencyDetails_ModelName", "ModelTransparencyDetails_ModelProviderName",
    "SensitivityLabelId", "ThreadId", "Workload",
]
for c in REQUIRED_TEXT_COLS:
    fact = ensure_col(fact, c)                    # text, null when absent

# Non-text contract columns are always produced above, but keep types explicit.
fact = fact.withColumn("CreationDate", F.col("CreationDate").cast("timestamp")) \
           .withColumn("InteractionDate", F.col("InteractionDate").cast("date")) \
           .withColumn("WeekStart", F.col("WeekStart").cast("date")) \
           .withColumn("MonthStart", F.col("MonthStart").cast("date")) \
           .withColumn("Resource_Count", F.col("Resource_Count").cast("long"))

missing = [c for c in (REQUIRED_TEXT_COLS + ["CreationDate","InteractionDate","WeekStart","MonthStart","Resource_Count"]) if c not in fact.columns]
assert not missing, f"model contract broken, missing: {missing}"
print("model contract satisfied: all 33 sourced columns present")

## Behaviour enrichment (was 26 DAX calculated columns)
Ported from the template's calculated columns so the curated table carries them and the
semantic model imports with **zero calculated columns** -> fast Import+Incremental Refresh
(Pro-safe) and Direct Lake-eligible. Runs after the contract guard, before the Delta write.

In [ ]:
# ----------------------------------------------------------------------------
# 10b. BEHAVIOUR ENRICHMENT  (ported from the .pbit DAX calculated columns)
#      Produces the 26 columns the semantic model used to compute at refresh
#      time, so the fact table imports with ZERO calculated columns:
#        - Import + Incremental Refresh stays fast (no per-row DAX)
#        - model becomes Direct Lake-eligible (calc columns block Direct Lake)
#      Row-level and deterministic -> identical output to the old DAX.
# ----------------------------------------------------------------------------
from pyspark.sql import Column, Window

def L(v):
    return v if isinstance(v, Column) else F.lit(v)

def chain(pairs, default):
    e = None
    for cond, val in pairs:
        e = F.when(cond, L(val)) if e is None else e.when(cond, L(val))
    return e.otherwise(L(default))

def _s(c):        # lower+trim, null -> ""   (for case-insensitive compares)
    return F.lower(F.trim(F.coalesce(F.col(c).cast("string"), F.lit(""))))

def _raw(c):      # trim, null -> ""         (case preserved)
    return F.trim(F.coalesce(F.col(c).cast("string"), F.lit("")))

def _notblank(c):
    return F.col(c).isNotNull() & (F.trim(F.col(c).cast("string")) != "")

def _clean2(c):   # trimmed value, or NULL when blank (for COALESCE fallbacks)
    t = F.trim(F.col(c).cast("string"))
    return F.when(t.isNotNull() & (t != ""), t)

def _has(colexpr, subs):
    e = F.lit(False)
    for s in subs:
        e = e | colexpr.contains(s)
    return e

def _in(colexpr, vals):     # colexpr already lowered; vals lowercased here
    return colexpr.isin(*[v.lower() for v in vals])

# --- lowered base expressions (reused) --------------------------------------
appHost   = _s("AppHost")
ctxType   = _s("Context_Type")
resType   = _s("AccessedResource_Type")
resAction = _s("AccessedResource_Action")
siteUrl   = _s("AccessedResource_SiteUrl")
pluginId  = _s("AISystemPlugin_Id")
isActive  = _has(resAction, ["send","draft","create","post","invoke","write","patch","execute"])
HYPER     = "http://schema.skype.com/hyperlink"

# --- Environment / License Status -------------------------------------------
hl    = F.upper(F.trim(F.coalesce(F.col("Has license").cast("string"), F.lit(""))))
isLic = hl.isin("YES","TRUE","Y","1")
fact = fact.withColumn("Environment", F.when(isLic, F.lit("Licensed")).otherwise(F.lit("Unlicensed")))
fact = fact.withColumn("License Status",
                       F.when(isLic, F.lit("M365 Copilot Licensed")).otherwise(F.lit("Unlicensed")))

# --- Is_Sensitive -----------------------------------------------------------
fact = fact.withColumn("Is_Sensitive",
                       (_notblank("SensitivityLabelId") | _notblank("AccessedResource_SensitivityLabelId")))

# --- AI_Model ---------------------------------------------------------------
mdl = F.upper(F.coalesce(F.col("ModelTransparencyDetails_ModelName").cast("string"), F.lit("")))
fact = fact.withColumn("AI_Model", chain([
    ((mdl == "") | (mdl == "NULL"), "Embedded App (no model logged)"),
    (mdl.contains("DEEP_LEO"), "GPT-4 (Standard)"),
    (mdl.contains("REASONING"), "Reasoning Model (o1/o3)"),
    (mdl.contains("OFFENSIVE"), "Safety Filter (blocked)"),
    (mdl.contains("GPT-41") | mdl.contains("GPT-4.1"), "GPT-4.1 (Next Gen)"),
    (mdl.contains("O3-MINI") | mdl.contains("O3MINI"), "o3-mini (Reasoning)"),
    (mdl.contains("O3") | mdl.contains("O1"), "Reasoning Model (o-series)"),
    (mdl.contains("GPT-5") | mdl.contains("GPT5"), "GPT-5 (Next Gen)"),
    (mdl.contains("CLAUDE"), "Claude (Anthropic)"),
    (mdl.contains("GEMINI"), "Gemini (Google)"),
    (mdl.contains("LLAMA") | mdl.contains("META"), "LLaMA (Meta)"),
    (mdl.contains("PHI"), "Phi (Microsoft Small Model)"),
], mdl))

# --- Behavior_Category ------------------------------------------------------
from_resource = chain([
    (_in(resAction, ["sendemailv2","draftemail","senddraftemail","updatedraftemail"]), "Email Drafting"),
    ((resType == "emailmessage") & isActive, "Email Drafting"),
    (resType == "emailmessage", "Email Summarising"),
    (resAction == "mcp_meetingmanagement", "Meeting Scheduling"),
    (_in(resType, ["event","teamsmeeting"]), "Meeting Prep"),
    (_in(resAction, ["postmessagetoconversation","createchat"]), "Teams Messaging"),
    (_in(resType, ["teamsmessage","teamschat","teamschannel"]), "Teams Messaging"),
    (resType == "flow", "Running a Workflow"),
    (_in(resType, ["connector","http"]) & isActive, "Running a Workflow"),
    (_in(resAction, ["executedatasetquery","getitems","getalltables","gettableviews"]), "Data Querying"),
    (_in(resType, ["xlsx","csv","xlsm","xlsb","xls"]),
        F.when(isActive, F.lit("Excel Assistance")).otherwise(F.lit("Spreadsheet Review"))),
    (resType == "peopleinferenceanswer", "People Lookup"),
    ((resType == "listitem") | (resType == "aspx"), "Enterprise Searching"),
    (resType == "websearchquery", "Web Searching"),
    (resType == "pdf", "PDF Analysis"),
    (_in(resType, ["py","js","java","tsx","jsx","css","php","sh"]) & isActive, "Code Writing"),
    (_in(resType, ["py","sql","js","java","json","xml","html","yaml","yml","txt"]), "Code Analysis"),
    (_in(resType, ["png","jpg","jpeg","svg","gif"]) & isActive, "Image Generation"),
    (_in(resType, ["png","jpg","jpeg","gif"]), "Image / Media Analysis"),
    (_in(resType, ["streamvideo","mp4","mov","webm","mkv"]), "Video Summarising"),
    (_in(resType, ["planid","taskids"]), "Task Management"),
    (resType == "looppage", "Real-time Collaboration"),
    ((resType == HYPER) & _has(siteUrl, ["github.com","stackoverflow.com","npmjs.com","pypi.org","docker.com","kubernetes.io","leetcode.com"]), "Code Analysis"),
    ((resType == HYPER) & _has(siteUrl, ["learning.cloud.microsoft","coursera.org","udemy.com"]), "Agent: Coaching"),
    ((resType == HYPER) & siteUrl.contains("sharepoint.com"), "Enterprise Searching"),
    (_in(resType, ["external","http"]) | (resType == HYPER), "Web Searching"),
    (_in(resType, ["docx","doc","rtf"]) & isActive, "Document Drafting"),
    (_in(resType, ["docx","doc","rtf"]) & (resAction == "read"), "File Retrieval"),
    (_in(resType, ["docx","doc","rtf"]), "Document Summarising"),
    (_in(resType, ["pptx","ppt","potx"]) & isActive, "Presentation Creation"),
    (_in(resType, ["pptx","ppt","potx"]) & (resAction == "read"), "File Retrieval"),
    (_in(resType, ["pptx","ppt","potx"]), "Presentation Summarising"),
    (_has(siteUrl, ["service-now.com","servicenow.com"]), "Agent: IT & Service Desk"),
    (siteUrl.contains("dynamics.com"), "Agent: Sales & Customer"),
], None)

from_plugin = F.when(pluginId == "enterprisesearch", F.lit("Enterprise Searching")).otherwise(F.lit(None).cast("string"))

from_context = chain([
    (ctxType == "teamsmeeting", "Meeting Prep"),
    (ctxType == "streamvideo", "Video Summarising"),
    (ctxType == "docx",
        F.when((appHost == "word") & isActive, F.lit("Document Drafting")).otherwise(F.lit("Document Summarising"))),
    (_in(ctxType, ["xlsx","xlsm","xlsb","xls","csv"]), "Spreadsheet Review"),
    (_in(ctxType, ["pptx","pptm"]),
        F.when((appHost == "powerpoint") & isActive, F.lit("Presentation Creation")).otherwise(F.lit("Presentation Summarising"))),
    (_in(ctxType, ["teamschat","teamschannel"]), "Teams Messaging"),
    (ctxType == "aspx", "Enterprise Searching"),
    ((appHost.isin("outlook","outlooksidepane")) & isActive, "Email Drafting"),
    (appHost.isin("outlook","outlooksidepane"), "Email Summarising"),
    (appHost == "excel", "Excel Assistance"),
    ((appHost == "word") & isActive, "Document Drafting"),
    (appHost == "word", "Document Summarising"),
    ((appHost == "powerpoint") & isActive, "Presentation Creation"),
    (appHost == "powerpoint", "Presentation Summarising"),
    (appHost == "stream", "Video Summarising"),
    (appHost == "sharepoint", "SharePoint Access"),
    (appHost == "designer", "Image Generation"),
    (appHost == "onenote", "Note Taking"),
    (appHost == "forms", "Form / Survey Work"),
    (appHost == "planner", "Task Management"),
    (_in(appHost, ["loop","whiteboard","vivaengage"]), "Real-time Collaboration"),
    (appHost == "copilot studio", "Domain-Specific Agent"),
    (appHost == "autonomous", "Running a Workflow"),
    ((appHost == "logic app") & (_notblank("AgentName") | _notblank("AgentId")), "Running a Workflow"),
    (_in(appHost, ["datawarehousing core","power bi"]), "Data Querying"),
], "General Chat")

fact = fact.withColumn("Behavior_Category", F.coalesce(from_resource, from_plugin, from_context))

# --- Behavior_Enriched ------------------------------------------------------
agentName_l = _s("AgentName")
bcat = F.col("Behavior_Category")
env  = F.col("Environment")
qna_set = ["General Q&A","M365 Chat Q&A","Teams Q&A","Browser Q&A","General Chat"]
fact = fact.withColumn("Behavior_Enriched", chain([
    ((env != "Agents") & (env != "Cowork"), bcat),
    (~bcat.isin(*qna_set), bcat),
    (_has(agentName_l, ["coach","mentor","learning","career"]), "Agent: Coaching"),
    (_has(agentName_l, ["research","analyst","analy"]), "Agent: Research & Analysis"),
    (_has(agentName_l, ["sales","commercial","customer","crm","revenue"]), "Agent: Sales & Customer"),
    (_has(agentName_l, ["hr","recruit","talent","onboard","people"]), "Agent: HR & People"),
    (_has(agentName_l, ["policy","compliance","legal","audit","risk"]), "Agent: Compliance & Policy"),
    (_has(agentName_l, ["service","support","help","ticket","incident"]), "Agent: IT & Service Desk"),
    (_has(agentName_l, ["summar","draft","translat","editor"]), "Agent: Content Generation"),
    (_has(agentName_l, ["data","report","dashboard","metric"]), "Agent: Data & Reporting"),
    (_has(agentName_l, ["knowledge","faq","wiki","buddy","guide"]), "Agent: Knowledge Base"),
    (_has(agentName_l, ["idea","brainstorm","creative","design"]), "Agent: Ideation & Creative"),
], "Agent: General Purpose"))

# --- Agents 365 lookup (Agent_LinkID -> Title ID) for the *_Full enrichment --
def _a365_lookup():
    try:
        a = spark.table(SRC_AGENTS)
    except Exception as e:
        print(f"[warn] {SRC_AGENTS} not found ({e}); A365 enrichment degrades to blank.")
        return None
    tid = first_existing(a, ["Title ID","Title Id","TitleID","Title ID "])
    if not tid:
        return None
    def pick(names):
        c = first_existing(a, names)
        return F.col(c).cast("string") if c else F.lit(None).cast("string")
    out = (a.withColumn("_a_title", F.trim(F.col(tid).cast("string")))
             .where(F.col("_a_title").isNotNull() & (F.col("_a_title") != ""))
             .withColumn("A365_Desc",   pick(["Agent description","Agent description ","Agent Description"]))
             .withColumn("A365_Actions",pick(["Custom actions","Custom actions ","Custom Actions"]))
             .withColumn("A365_Code",   pick(["Can use code interpreter","Can use code interpreter "]))
             .withColumn("A365_Images", pick(["Can generate images using user prompt","Can generate images using user prompt "]))
             .withColumn("A365_SP",     pick(["Can read Sharepoint sites and files","Can read Sharepoint sites and files "]))
             .withColumn("A365_Type",   pick(["Agent type (A365)","Agent type (A365) ","Agent Type (A365)"]))
             .select("_a_title","A365_Desc","A365_Actions","A365_Code","A365_Images","A365_SP","A365_Type")
             .dropDuplicates(["_a_title"]))
    return out

_a365 = _a365_lookup()
if _a365 is not None:
    fact = fact.join(F.broadcast(_a365), fact["Agent_LinkID"] == _a365["_a_title"], "left").drop("_a_title")
for _c in ["A365_Desc","A365_Actions","A365_Code","A365_Images","A365_SP","A365_Type"]:
    fact = ensure_col(fact, _c)

# --- Behavior_Enriched_Full -------------------------------------------------
be   = F.col("Behavior_Enriched")
needs = (env == "Agents") & (be == "Agent: General Purpose")
desc    = F.when(needs, F.lower(F.coalesce(F.col("A365_Desc").cast("string"), F.lit("")))).otherwise(F.lit(""))
actions = F.when(needs, F.lower(F.coalesce(F.col("A365_Actions").cast("string"), F.lit("")))).otherwise(F.lit(""))
search  = F.concat_ws(" ", desc, actions)
hasCode   = needs & (F.col("A365_Code")   == "Yes")
hasImages = needs & (F.col("A365_Images") == "Yes")
hasSP     = needs & (F.col("A365_SP")     == "Yes")
resolved = chain([
    (~needs, be),
    (_has(search, ["coach","mentor","learning","training","skill"]), "Agent: Coaching"),
    (_has(search, ["research","analyst","analy","insight","intelligence"]), "Agent: Research & Analysis"),
    (_has(search, ["sales","commercial","customer","crm","pipeline","prospect","deal"]), "Agent: Sales & Customer"),
    (_has(search, ["recruit","talent","onboard","hiring","employee","human resource","job description"]), "Agent: HR & People"),
    (_has(search, ["policy","compliance","legal","audit","risk","governance"]), "Agent: Compliance & Policy"),
    (_has(search, ["support","helpdesk","troubleshoot","ticket","incident","service desk"]), "Agent: IT & Service Desk"),
    (_has(search, ["summar","draft","translat","content","communications"]), "Agent: Content Generation"),
    (_has(search, ["data","report","dashboard","analytics","metric"]), "Agent: Data & Reporting"),
    (_has(search, ["knowledge","faq","wiki","guide","handbook","documentation"]), "Agent: Knowledge Base"),
    (_has(search, ["brainstorm","creative","design","innovat"]), "Agent: Ideation & Creative"),
    (hasCode, "Agent: Data & Reporting"),
    (hasImages, "Agent: Ideation & Creative"),
    (hasSP, "Agent: Knowledge Base"),
], "Agent: General Purpose")
fact = fact.withColumn("_Resolved", resolved)
wfsig = F.lower(F.concat_ws(" | ",
    F.coalesce(F.col("AgentName").cast("string"), F.lit("")),
    F.coalesce(F.col("AccessedResource_SiteUrl").cast("string"), F.lit("")),
    F.coalesce(F.col("AccessedResource_Action").cast("string"), F.lit("")),
    F.coalesce(F.col("AppHost").cast("string"), F.lit(""))))
fact = fact.withColumn("Behavior_Enriched_Full", chain([
    (F.col("_Resolved") != "Running a Workflow", F.col("_Resolved")),
    (_has(wfsig, ["servicenow","salesforce","dynamics","workday","jira","zendesk","service desk","servicedesk"]), "Specialist / Line-of-Business Workflow"),
    (_has(wfsig, ["outlook","exchange","mail"]), "Email Workflow"),
    (_has(wfsig, ["calendar","meeting","schedul"]), "Meeting Workflow"),
    (_has(wfsig, ["power bi","powerbi","dataverse","dataset","report","dashboard","excel","sql","analytics"]), "Data & Reporting Workflow"),
    (_has(wfsig, ["sharepoint","onedrive","word","document",".doc","file"]), "Document Workflow"),
    (_has(wfsig, ["planner","task","approv","teams","notify","post","list"]), "Coordination Workflow"),
], "General Workflow")).drop("_Resolved")

# --- Behavior_Source --------------------------------------------------------
agent_r  = _raw("AgentName")
plugin_r = _raw("AISystemPlugin_Name")
app_r    = _raw("AppHost")
src_expr = chain([
    (env == "Cowork", F.concat(F.lit("Cowork"),
        F.when(agent_r != "", F.concat(F.lit(": "), agent_r)).otherwise(F.lit("")))),
    ((env == "Agents") & (agent_r != ""), F.concat(F.lit("Agent: "), agent_r)),
    (plugin_r != "", F.concat(app_r, F.lit(" ("), plugin_r, F.lit(")"))),
    (app_r != "", app_r),
], "Copilot Chat")
fact = fact.withColumn("Behavior_Source", F.concat(bcat, F.lit(" \u2192 "), src_expr))

# --- Value_Outcome ----------------------------------------------------------
isSens = F.col("Is_Sensitive")
fact = fact.withColumn("Value_Outcome", chain([
    (be.isin("Email Summarising","Email Triage","Email Thread Summary"), "Time Saved (Email)"),
    (be.isin("Meeting Prep","Video Summarising"), "Time Saved (Meetings)"),
    (be.isin("Document Summarising","Presentation Summarising","Note Taking"), "Time Saved (Documents)"),
    (be.isin("Web Searching","Enterprise Searching","File Retrieval","PDF Analysis","SharePoint Access","People Lookup","Agent: Knowledge Base"), "Search Time Saved"),
    (be.isin("Teams Messaging","Meeting Scheduling"), "Communication Time Saved"),
    (be.isin("Spreadsheet Review","Spreadsheet Analysis","Excel Assistance"), "Spreadsheet Time Saved"),
    (be.isin("Email Drafting","Document Drafting","Presentation Creation","Image Generation","Image / Media Analysis","Image/Media Analysis","Agent: Content Generation","Agent: Ideation & Creative"), "Content Output"),
    (be.isin("Real-time Collaboration","Form / Survey Work"), "Team Collaboration"),
    ((be == "Running a Workflow") | (env == "Cowork"), "Workflow Automation"),
    (be == "Task Management", "Task Coordination"),
    (isSens & (env != "Agents") & (env != "Cowork"), "Compliance & Risk"),
    (be.isin("Data Querying","Agent: Data & Reporting","Agent: Research & Analysis"), "Data-Driven Decisions"),
    (be.isin("Code Writing","Code Analysis","Code Analysis (URL)"), "Coding Capability"),
    (be.isin("Agent: Coaching","Agent: Coaching (URL)"), "Skills Development"),
    (be == "Agent: Sales & Customer", "Revenue Enablement"),
    (be == "Agent: IT & Service Desk", "Service Desk Deflection"),
    (be == "Agent: Compliance & Policy", "Compliance & Risk"),
    (be == "Agent: HR & People", "HR Expertise"),
    (be.isin("Domain-Specific Agent","Cross-Org Agent"), "Specialist Expertise"),
], "General AI Productivity"))

# --- Usage_Mode -------------------------------------------------------------
bef = F.col("Behavior_Enriched_Full")
a365type = F.coalesce(F.col("A365_Type").cast("string"), F.lit(""))
isDelegating = (env == "Cowork") | (appHost == "autonomous") | bef.contains("Workflow") | a365type.isin("Autonomous","Workflow","Triggered")
producing = ["Email Drafting","Document Drafting","Presentation Creation","Image Generation","Code Writing","Code Analysis","Code Analysis (URL)","Data Querying","Spreadsheet Analysis","Excel Assistance","Agent: Content Generation","Agent: Ideation & Creative","Agent: Research & Analysis","Agent: Data & Reporting","Agent: Sales & Customer","Agent: HR & People","Agent: IT & Service Desk","Agent: Compliance & Policy","Agent: Coaching","Agent: Coaching (URL)","Domain-Specific Agent","Cross-Org Agent","Form / Survey Work","Real-time Collaboration","Note Taking","Teams Messaging","Meeting Scheduling","Task Management"]
consuming = ["Document Summarising","Email Summarising","Email Thread Summary","Email Triage","Presentation Summarising","Video Summarising","Meeting Prep","Image / Media Analysis","Image/Media Analysis","Sensitive Content Interaction"]
finding   = ["Web Searching","Enterprise Searching","PDF Analysis","SharePoint Access","File Retrieval","People Lookup","Agent: Knowledge Base","Spreadsheet Review"]
fact = fact.withColumn("Usage_Mode", chain([
    (isDelegating, "5 - Delegating"),
    (bef.isin(*producing), "4 - Producing"),
    (bef.isin(*consuming), "3 - Consuming"),
    (bef.isin(*finding), "2 - Finding"),
], "1 - Asking"))

# --- Expertise_Role ---------------------------------------------------------
fact = fact.withColumn("Expertise_Role", chain([
    (bef.isin("Data Querying","Agent: Data & Reporting","Spreadsheet Analysis"), "Data Analyst"),
    (bef.isin("Code Writing","Code Analysis","Code Analysis (URL)"), "Software Engineer"),
    (bef.isin("Agent: Research & Analysis"), "Business Analyst"),
    (bef.isin("Agent: Compliance & Policy","Sensitive Content Interaction"), "Compliance Specialist"),
    (bef.isin("Agent: Sales & Customer"), "Sales Consultant"),
    (bef.isin("Agent: IT & Service Desk"), "IT Specialist"),
    (bef.isin("Agent: HR & People"), "HR Specialist"),
    (bef.isin("Agent: Coaching","Agent: Coaching (URL)"), "Coach"),
    (bef.contains("Workflow") | (bef == "Task Management"), "Automation Engineer"),
    (bef.isin("Domain-Specific Agent","Cross-Org Agent"), "Domain Expert"),
    (bef.isin("Email Drafting"), "Communications Specialist"),
    (bef.isin("Email Triage","Meeting Scheduling","Email Summarising","Email Thread Summary"), "Executive Assistant"),
    (bef.isin("Document Drafting","Agent: Content Generation","Note Taking","Document Summarising"), "Content Writer"),
    (bef.isin("Presentation Creation","Presentation Summarising"), "Presentation Designer"),
    (bef.isin("Image Generation","Image/Media Analysis","Image / Media Analysis","Agent: Ideation & Creative"), "Visual Designer"),
    (bef.isin("Meeting Prep","Video Summarising"), "Meeting Coordinator"),
    (bef.isin("Web Searching","PDF Analysis","Agent: Knowledge Base"), "Researcher"),
    (bef.isin("Enterprise Searching","SharePoint Access","File Retrieval","People Lookup"), "Knowledge Navigator"),
    (bef.isin("Spreadsheet Review","Excel Assistance"), "Spreadsheet Specialist"),
    (bef.isin("Real-time Collaboration","Form / Survey Work","Form/Survey Work","Teams Messaging"), "Collaboration Lead"),
], None))

# --- Efficiency_Breakdown ---------------------------------------------------
fact = fact.withColumn("Efficiency_Breakdown", chain([
    (bef.isin("Email Summarising","Email Triage","Email Thread Summary","Email Drafting"), "Email"),
    (bef.isin("Document Summarising","Note Taking","Document Drafting","Agent: Content Generation"), "Document Assistance"),
    (bef.isin("Presentation Summarising","Presentation Creation"), "Presentations"),
    (bef.isin("Meeting Prep","Video Summarising","Meeting Scheduling"), "Meetings"),
    (bef.isin("Web Searching","Enterprise Searching","PDF Analysis","SharePoint Access","File Retrieval","People Lookup","Agent: Knowledge Base","Agent: Research & Analysis"), "Search & Research"),
    (bef.isin("Spreadsheet Review","Excel Assistance","Spreadsheet Analysis","Data Querying","Agent: Data & Reporting"), "Data & Spreadsheets"),
    (bef.isin("Image Generation","Image / Media Analysis","Image/Media Analysis","Agent: Ideation & Creative","Code Writing","Code Analysis","Code Analysis (URL)"), "Creative & Technical"),
    (bef.isin("Teams Messaging","Real-time Collaboration","Form / Survey Work","Task Management") | bef.contains("Workflow"), "Collaboration & Workflows"),
    (bef.isin("Agent: Sales & Customer","Agent: IT & Service Desk","Agent: HR & People","Agent: Compliance & Policy","Agent: Coaching","Agent: Coaching (URL)","Domain-Specific Agent","Cross-Org Agent"), "Specialist Agents"),
    (bcat == "Teams Q&A", "Teams Chat"),
    (bcat == "M365 Chat Q&A", "BizChat Q&A"),
    (bcat == "Browser Q&A", "BizChat Q&A"),
], "General Q&A"))

# --- Web_Grounded_Signal ----------------------------------------------------
isInternal = _has(siteUrl, ["sharepoint.com",".onmicrosoft.com"])
fact = fact.withColumn("Web_Grounded_Signal",
    F.when((resType == "websearchquery") | _in(resType, ["external","http"]) | ((resType == HYPER) & ~isInternal),
           F.lit("Web Grounded")).otherwise(F.lit("Not Web Grounded")))

# --- Behavior_Plausible -----------------------------------------------------
lic = F.col("License Status")
unlic_ok = ["General Chat","Web Searching","PDF Analysis","Document Summarising","Image / Media Analysis","Image Generation","Code Analysis","Translation"]
plausible_switch = chain([
    (bcat.isin("Email Summarising","Email Drafting"), "Free Chat Workaround (pasting Email)"),
    (bcat.isin("Excel Assistance","Spreadsheet Review","Data Querying"), "Free Chat Workaround (pasting Spreadsheet/Data)"),
    (bcat.isin("Meeting Prep","Meeting Scheduling"), "Free Chat Workaround (pasting Meeting info)"),
    (bcat == "Teams Messaging", "Free Chat Workaround (pasting Teams content)"),
    (bcat.isin("Enterprise Searching","People Lookup"), "Free Chat Workaround (pasting Enterprise data)"),
    (bcat.isin("Running a Workflow","Task Management"), "Free Chat Workaround (pasting Workflow)"),
    (bcat == "Real-time Collaboration", "Free Chat Workaround (pasting Loop content)"),
    (bcat == "Code Writing", "Free Chat Workaround (pasting Code)"),
    (bcat == "Video Summarising", "Free Chat Workaround (uploading Video)"),
], "Free Chat Workaround (Other)")
fact = fact.withColumn("Behavior_Plausible",
    F.when((lic == "M365 Copilot Licensed") | bcat.isin(*unlic_ok), bcat).otherwise(plausible_switch))

# --- Workflow_Action --------------------------------------------------------
fact = fact.withColumn("Workflow_Action",
    F.when(~bef.contains("Workflow"), F.lit("")).otherwise(chain([
        (_has(resAction, ["send","post","notify"]), "Sending / Notifying"),
        (_has(resAction, ["create","draft","write","add"]), "Creating Content"),
        (_has(resAction, ["invoke","execute","trigger","run"]), "Invoking / Triggering"),
        (_has(resAction, ["update","patch","modify","set"]), "Updating Records"),
        (_has(resAction, ["read","get","list","fetch"]), "Reading Data"),
        (_has(resAction, ["delete","remove"]), "Deleting / Removing"),
        (appHost == "autonomous", "Autonomous Run (no action logged)"),
        (appHost == "logic app", "Logic App Run (no action logged)"),
    ], "Workflow (other)")))

# --- Is_Agent_Activity ------------------------------------------------------
isAutonomous = appHost.isin("autonomous","logic app") | resType.isin("flow","connector")
fact = fact.withColumn("Is_Agent_Activity", (_notblank("AgentName") | _notblank("AgentId") | isAutonomous))

# --- Agent Filter -----------------------------------------------------------
isCowork = _s("AppHost").contains("cowork") | _s("Environment").contains("cowork")
fact = fact.withColumn("Agent Filter",
    F.when(isCowork, F.lit("Cowork"))
     .when(F.col("Is_Agent_Activity") == True, F.lit("Agents"))
     .otherwise(F.lit(None).cast("string")))

# --- Grounding Source -------------------------------------------------------
plugin = F.lower(F.concat_ws(" ",
    F.coalesce(F.col("AISystemPlugin_Id").cast("string"), F.lit("")),
    F.coalesce(F.col("AISystemPlugin_Name").cast("string"), F.lit(""))))
usedWeb = _has(plugin, ["bing","web"])
usedInternal = (F.col("Resource_Count") > 0) | _notblank("Context_Type")
fact = fact.withColumn("Grounding Source", chain([
    (usedWeb & usedInternal, "Mixed (Internal+Web)"),
    (usedWeb, "External (Web)"),
    (usedInternal, "Internal"),
], "Ungrounded"))

# --- Agent_Surface ----------------------------------------------------------
host = appHost
agent_l = agentName_l
fact = fact.withColumn("Agent_Surface", chain([
    (host.contains("cowork") | agent_l.contains("cowork"), "Cowork"),
    (agent_l.contains("scout"), "Scout"),
    (host.isin("autonomous","logic app") | resType.isin("flow","connector"), "Autonomous / Flow"),
    (F.col("Is_Agent_Activity") == True, "Copilot Agents"),
], None))

# --- Execution_Trigger ------------------------------------------------------
fact = fact.withColumn("Execution_Trigger",
    F.when(host.isin("autonomous","logic app") | resType.isin("flow","connector"),
           F.lit("Scheduled / Autonomous")).otherwise(F.lit("Interactive / On-demand")))

# --- UserMonthKey / Delegation_Event_Key / ActivityDate ---------------------
fact = fact.withColumn("UserMonthKey",
    F.concat(F.coalesce(F.col("Audit_UserId").cast("string"), F.lit("")), F.lit("|"),
             F.date_format(F.col("MonthStart"), "yyyy-MM")))
fact = fact.withColumn("Delegation_Event_Key",
    F.concat(F.coalesce(F.col("Audit_UserId").cast("string"), F.lit("")), F.lit("|"),
             F.date_format(F.col("InteractionDate"), "yyyy-MM-dd"), F.lit("|"),
             F.coalesce(_clean2("AgentName"), _clean2("Workflow_Action"), _clean2("AppHost"), F.lit("unknown-workflow"))))
fact = fact.withColumn("ActivityDate", F.col("InteractionDate").cast("timestamp"))

# --- Agent Last Used Date  (per-AgentName max CreationDate) ------------------
w_agent = Window.partitionBy("AgentName")
fact = fact.withColumn("Agent Last Used Date",
    F.when(_notblank("AgentName"), F.max("CreationDate").over(w_agent)).otherwise(F.lit(None).cast("timestamp")))

# --- User_Stage_Maturity / User_Stage  (UserMonthMetrics staging) -----------
umm = (fact.groupBy("Audit_UserId", "MonthStart")
    .agg(
        F.countDistinct("Behavior_Enriched_Full").alias("BehaviorCount"),
        F.max(F.when(_notblank("AgentName"), F.lit(1)).otherwise(F.lit(0))).alias("_HasAgent"),
        F.countDistinct("InteractionDate").alias("ActiveDays"),
        F.sum(F.when(F.col("Usage_Mode").isin("4 - Producing","5 - Delegating"), F.lit(1)).otherwise(F.lit(0))).alias("_ValueRows"),
        F.count(F.lit(1)).alias("_TotalRows"),
    ))
umm = umm.withColumn("HasAgent", F.col("_HasAgent") > 0)
umm = umm.withColumn("ValueFocusShare",
    F.when(F.col("_TotalRows") > 0, F.col("_ValueRows") / F.col("_TotalRows")).otherwise(F.lit(0.0)))
umm = umm.withColumn("UserStage", chain([
    ((F.col("ActiveDays") >= 15) | ((F.col("ActiveDays") >= 10) & (F.col("ValueFocusShare") >= 0.30) & F.col("HasAgent")), "4 - Power"),
    ((F.col("ActiveDays") >= 8) & (F.col("BehaviorCount") >= 5), "3 - Habitual"),
    ((F.col("ActiveDays") >= 3) & (F.col("BehaviorCount") >= 3), "2 - Developing"),
], "1 - Beginner"))
umm = umm.withColumn("_UMKey",
    F.concat(F.coalesce(F.col("Audit_UserId").cast("string"), F.lit("")), F.lit("|"),
             F.date_format(F.col("MonthStart"), "yyyy-MM"))).select("_UMKey", "UserStage")
fact = fact.join(F.broadcast(umm), fact["UserMonthKey"] == umm["_UMKey"], "left").drop("_UMKey")
fact = fact.withColumnRenamed("UserStage", "User_Stage_Maturity")
fact = fact.withColumn("User_Stage", F.col("User_Stage_Maturity"))

# --- drop A365 helper columns (not part of the model contract) --------------
fact = fact.drop("A365_Desc","A365_Actions","A365_Code","A365_Images","A365_SP","A365_Type")

ENRICHED_COLS = ["Environment","License Status","Is_Sensitive","AI_Model","Behavior_Category",
    "Behavior_Enriched","Behavior_Enriched_Full","Behavior_Source","Value_Outcome","Usage_Mode",
    "Expertise_Role","Efficiency_Breakdown","Web_Grounded_Signal","Behavior_Plausible","Workflow_Action",
    "Is_Agent_Activity","Agent Filter","Grounding Source","Agent_Surface","Execution_Trigger",
    "UserMonthKey","Delegation_Event_Key","ActivityDate","Agent Last Used Date",
    "User_Stage_Maturity","User_Stage"]
_missing_enr = [c for c in ENRICHED_COLS if c not in fact.columns]
assert not _missing_enr, f"enrichment incomplete, missing: {_missing_enr}"
print(f"enrichment complete: +{len(ENRICHED_COLS)} columns (fact now {len(fact.columns)} cols)")


In [ ]:
# ----------------------------------------------------------------------------
# 11. WRITE CURATED DELTA  (Power BI reads this table verbatim)
# ----------------------------------------------------------------------------
print(f"curated: {len(fact.columns)} cols -> {OUT_TABLE}  (mode={WRITE_MODE})")

if WRITE_MODE == "merge" and spark.catalog.tableExists(OUT_TABLE) and all(k in fact.columns for k in MERGE_KEYS):
    from delta.tables import DeltaTable
    tgt = DeltaTable.forName(spark, OUT_TABLE)
    cond = " AND ".join([f"t.`{k}` = s.`{k}`" for k in MERGE_KEYS])
    (tgt.alias("t").merge(fact.alias("s"), cond)
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
else:
    (fact.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .option("delta.columnMapping.mode", "name")
        .option("delta.minReaderVersion", "2")
        .option("delta.minWriterVersion", "5")
        .format("delta").saveAsTable(OUT_TABLE))

if RUN_OPTIMIZE:
    spark.sql(f"OPTIMIZE {OUT_TABLE} VORDER")   # compact + V-Order for Direct Lake / fast import

print("done. row count:", spark.table(OUT_TABLE).count())

## Point Power BI at the curated table

In the template, the `Chat + Agent Interactions (Audit Logs)` partition M becomes a thin,
**foldable** passthrough (no JSON parse, no expand, no joins):

```m
let
    Source   = FabricTable("copilot_interactions_curated"),
    Filtered = Table.SelectRows(Source, each [CreationDate] >= RangeStart and [CreationDate] < RangeEnd)
in
    Filtered
```

Because the only step is a range filter on `CreationDate`, it folds to the Lakehouse SQL
endpoint and Incremental Refresh only touches new day-partitions. On Fabric / Premium
capacity you can instead flip the model to **Direct Lake** and skip refresh entirely.